# Purview Migration Toolkit - Fabric Lakehouse Workflow

This notebook orchestrates Microsoft Purview artifact migration between tenants, storing all data and reports in the Lakehouse Files area.

**Prerequisites:**
- Access to source and target Purview accounts
- Azure CLI logged in or managed identity available
- Python 3.10+
- Fabric Lakehouse with Files area enabled

**Workflow:**
1. Export source Purview artifacts to manifest
2. Import into target Purview (dry-run + apply)
3. Generate relink plan
4. Apply relink plan (dry-run + apply)
5. Generate status reports (CSV/JSON)
6. Review results in Files area

## Section 1: Configure Lakehouse File Paths

Define the base path for the Lakehouse Files area and subdirectories for organizing manifests, plans, and reports.

In [ ]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime, timezone

# Lakehouse Files base path
LAKEHOUSE_FILES = "/lakehouse/default/Files"
LAKEHOUSE_ROOT = os.path.join(LAKEHOUSE_FILES, "purview_migration")

# Create subdirectory structure
MANIFESTS_DIR = os.path.join(LAKEHOUSE_ROOT, "manifests")
PLANS_DIR = os.path.join(LAKEHOUSE_ROOT, "plans")
REPORTS_DIR = os.path.join(LAKEHOUSE_ROOT, "reports")
LOGS_DIR = os.path.join(LAKEHOUSE_ROOT, "logs")

# New Lakehouse analytics package directories
LAKEHOUSE_PACKAGE_DIR = os.path.join(LAKEHOUSE_ROOT, "lakehouse_package")
LAKEHOUSE_JSON_DIR = os.path.join(LAKEHOUSE_PACKAGE_DIR, "json")
LAKEHOUSE_TABLES_DIR = os.path.join(LAKEHOUSE_PACKAGE_DIR, "tables")
LAKEHOUSE_SEMANTIC_DIR = os.path.join(LAKEHOUSE_PACKAGE_DIR, "semantic")

# File paths for key artifacts
SOURCE_MANIFEST_PATH = os.path.join(MANIFESTS_DIR, "source-export.json")
TARGET_STATUS_PATH = os.path.join(MANIFESTS_DIR, "target-import-status.json")
RELINK_PLAN_PATH = os.path.join(PLANS_DIR, "relink-plan.json")
RELINK_STATUS_PATH = os.path.join(PLANS_DIR, "relink-status.json")
REPORT_JSON_PATH = os.path.join(REPORTS_DIR, "relink-report.json")
REPORT_CSV_PREFIX = os.path.join(REPORTS_DIR, "relink-report")

print("Configured Lakehouse file paths:")
print(f"  Root: {LAKEHOUSE_ROOT}")
print(f"  Manifests: {MANIFESTS_DIR}")
print(f"  Plans: {PLANS_DIR}")
print(f"  Reports: {REPORTS_DIR}")
print(f"  Lakehouse package: {LAKEHOUSE_PACKAGE_DIR}")

## Section 2: Initialize Lakehouse Connection

Verify the Lakehouse connection and Files area accessibility using NotebookUtils.

In [ ]:
try:
    import mssparkutils

    # Verify Lakehouse is accessible
    _ = mssparkutils.fs.ls(LAKEHOUSE_FILES)
    print("Lakehouse connection verified")
    print(f"Files area accessible at: {LAKEHOUSE_FILES}")

    # Get runtime context
    import notebookutils

    context = notebookutils.runtime.context
    print(f"Notebook context available: {context is not None}")

except Exception as e:
    print(f"Warning: Lakehouse context not fully available: {e}")
    print("This is normal outside Fabric environment. Using standard file I/O.")

## Section 3: Create Directory Structure in Files Area

Create the required folder hierarchy for organizing migration artifacts.

In [ ]:
def ensure_dirs():
    """Create directory structure using pathlib."""
    for dir_path in [
        MANIFESTS_DIR,
        PLANS_DIR,
        REPORTS_DIR,
        LOGS_DIR,
        LAKEHOUSE_PACKAGE_DIR,
        LAKEHOUSE_JSON_DIR,
        LAKEHOUSE_TABLES_DIR,
        LAKEHOUSE_SEMANTIC_DIR,
    ]:
        Path(dir_path).mkdir(parents=True, exist_ok=True)
        print(f"Ensured directory: {dir_path}")

ensure_dirs()

# Try to list root directory to verify
try:
    files = mssparkutils.fs.ls(LAKEHOUSE_FILES)
    print(f"\nFiles area contents ({len(files)} items):")
    for f in files[:5]:
        print(f"  {f.name}")
    if len(files) > 5:
        print(f"  ... and {len(files) - 5} more")
except Exception as e:
    print(f"Note: Unable to list files with mssparkutils (OK outside Fabric): {type(e).__name__}")

## Section 4: Install and Initialize Purview Migration Toolkit

Install the Python migration toolkit from the repository.

In [ ]:
import subprocess

# Install purview-migration-toolkit from local repo or git
print("Installing Purview Migration Toolkit...")
try:
    result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/Edcorcor/Purview-Migration.git@main",
        ],
        capture_output=True,
        text=True,
        timeout=180,
    )
    if result.returncode != 0:
        print("Attempting local installation from current environment...")
        repo_root = Path.cwd().parent.parent
        if (repo_root / "src" / "purview_migration").exists():
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-e", str(repo_root)],
                capture_output=True,
                timeout=180,
            )
except Exception as e:
    print(f"Toolkit install warning: {e}")

# Import toolkit modules
try:
    from purview_migration.exporter import export_manifest
    from purview_migration.importer import import_manifest
    from purview_migration.relink import build_relink_plan
    from purview_migration.relink_executor import apply_relink_plan
    from purview_migration.report_generator import export_report
    from purview_migration.io_utils import read_json, write_json
    from purview_migration.lakehouse_export import package_manifest_for_lakehouse

    print("Purview Migration Toolkit imported successfully")
except ImportError as e:
    print(f"Failed to import toolkit: {e}")
    print("Ensure toolkit is installed or available in Python path")

## Configuration: Set Your Purview Account Names

Edit these values before running the migration workflow.

In [ ]:
# ============================================================================
# CONFIGURATION: Edit these values for your migration
# ============================================================================

SOURCE_PURVIEW_ACCOUNT = "source-purview-account"  # e.g., "my-prod-purview"
TARGET_PURVIEW_ACCOUNT = "target-purview-account"  # e.g., "my-adlz-purview"

# Export and validation settings
MAX_EXPORT_ENTITIES = 5000
MAX_ENTITY_VALIDATION = 2000

# Lakehouse packaging options
ENABLE_LAKEHOUSE_PACKAGE = True
INCLUDE_TABLE_EXPORTS = True
INCLUDE_SEMANTIC_REPORT = True
CREATE_LAKEHOUSE_TABLES_FROM_CSV = True
LAKEHOUSE_TABLE_PREFIX = "purview_migration_"

# Workflow execution flags
RUN_IMPORT_DRY_RUN = True
RUN_IMPORT_APPLY = False
RUN_RELINK_DRY_RUN = True
RUN_RELINK_APPLY = False
GENERATE_REPORTS = True
REPORT_FORMAT = "both"  # "json", "csv", or "both"

print("=" * 70)
print("PURVIEW MIGRATION CONFIGURATION")
print("=" * 70)
print(f"Source Purview Account:       {SOURCE_PURVIEW_ACCOUNT}")
print(f"Target Purview Account:       {TARGET_PURVIEW_ACCOUNT}")
print(f"Max Export Entities:          {MAX_EXPORT_ENTITIES}")
print(f"Max Entity Validation:        {MAX_ENTITY_VALIDATION}")
print(f"Lakehouse Package:            {ENABLE_LAKEHOUSE_PACKAGE}")
print(f"Include Table Exports:        {INCLUDE_TABLE_EXPORTS}")
print(f"Include Semantic Report:      {INCLUDE_SEMANTIC_REPORT}")
print(f"Create Lakehouse Tables:      {CREATE_LAKEHOUSE_TABLES_FROM_CSV}")
print(f"Lakehouse Table Prefix:       {LAKEHOUSE_TABLE_PREFIX}")
print(f"Import Dry-Run:               {RUN_IMPORT_DRY_RUN}")
print(f"Import Apply:                 {RUN_IMPORT_APPLY}")
print(f"Relink Dry-Run:               {RUN_RELINK_DRY_RUN}")
print(f"Relink Apply:                 {RUN_RELINK_APPLY}")
print(f"Generate Reports:             {GENERATE_REPORTS} ({REPORT_FORMAT})")
print("=" * 70)

## Step 1: Export Source Purview Artifacts and Build Lakehouse Package

Export source artifacts to a manifest, then create Lakehouse analytics outputs:
- JSON files by artifact type
- Table CSVs for assets, scans, collections, and related objects
- Semantic model and report specification files

In [ ]:
print("\n" + "=" * 70)
print("STEP 1: EXPORT SOURCE PURVIEW ARTIFACTS")
print("=" * 70)

try:
    print(f"\nExporting from: {SOURCE_PURVIEW_ACCOUNT}")
    manifest = export_manifest(SOURCE_PURVIEW_ACCOUNT, max_entities=MAX_EXPORT_ENTITIES)

    # Write manifest to Lakehouse
    manifest_dict = manifest.to_dict()
    write_json(SOURCE_MANIFEST_PATH, manifest_dict)
    print(f"Manifest exported to: {SOURCE_MANIFEST_PATH}")

    # Display summary
    print("\nExport summary:")
    print(f"  Collections:         {len(manifest.collections)}")
    print(f"  Data Sources:        {len(manifest.data_sources)}")
    print(f"  Scans:               {sum(len(x) for x in manifest.scans_by_source.values())}")
    print(f"  Glossary Categories: {len(manifest.glossary_categories)}")
    print(f"  Glossary Terms:      {len(manifest.glossary_terms)}")
    print(f"  Classifications:     {len(manifest.classifications)}")
    print(f"  Scan Rulesets:       {len(manifest.scan_rulesets)}")
    print(f"  Scan Credentials:    {len(manifest.scan_credentials)}")
    print(f"  Entities:            {len(manifest.entities)}")

    if manifest.warnings:
        print(f"\n{len(manifest.warnings)} warnings during export:")
        for w in manifest.warnings[:5]:
            print(f"  - {w}")
        if len(manifest.warnings) > 5:
            print(f"  ... and {len(manifest.warnings) - 5} more")
    else:
        print("\nNo warnings during export")

    if ENABLE_LAKEHOUSE_PACKAGE:
        print("\nBuilding Lakehouse package (json/tables/semantic)...")
        package_outputs = package_manifest_for_lakehouse(
            manifest_dict,
            LAKEHOUSE_PACKAGE_DIR,
            include_tables=INCLUDE_TABLE_EXPORTS,
            include_semantic_model=INCLUDE_SEMANTIC_REPORT,
        )
        print("Lakehouse package created")
        print(f"  Output dir: {package_outputs['outputDir']}")
        print(f"  JSON files: {len(package_outputs['jsonOutputs'])}")
        print(f"  Table files: {len(package_outputs['tableOutputs'])}")
        print(f"  Semantic/report files: {len(package_outputs['semanticOutputs'])}")
    else:
        print("\nLakehouse package generation disabled")

except Exception as e:
    print(f"Export failed: {e}")
    import traceback

    traceback.print_exc()

In [ ]:
print("\n" + "=" * 70)
print("STEP 1B: MATERIALIZE LAKEHOUSE TABLES")
print("=" * 70)

if not CREATE_LAKEHOUSE_TABLES_FROM_CSV:
    print("Table materialization disabled")
    print("Set CREATE_LAKEHOUSE_TABLES_FROM_CSV = True in the configuration cell to enable")
else:
    try:
        table_dir = Path(LAKEHOUSE_TABLES_DIR)
        if not table_dir.exists():
            raise FileNotFoundError(f"Table export directory not found: {table_dir}")

        csv_files = sorted(table_dir.glob("*.csv"))
        if not csv_files:
            raise FileNotFoundError("No CSV files found. Run Step 1 with table exports enabled first.")

        print(f"Found {len(csv_files)} CSV files in: {table_dir}")

        created_tables = []
        skipped_files = []

        for csv_file in csv_files:
            # Keep managed table names simple and deterministic.
            base_name = csv_file.stem.replace("-", "_").replace(" ", "_")
            table_name = f"{LAKEHOUSE_TABLE_PREFIX}{base_name}".lower()

            # CSV files include headers and mixed shape payload JSON; infer schema for convenience.
            df = (
                spark.read.format("csv")
                .option("header", True)
                .option("multiLine", True)
                .option("escape", '"')
                .option("quote", '"')
                .option("inferSchema", True)
                .load(str(csv_file))
            )

            if df.rdd.isEmpty():
                skipped_files.append(csv_file.name)
                print(f"Skipped empty file: {csv_file.name}")
                continue

            # Overwrite keeps reruns idempotent during notebook development.
            (
                df.write.mode("overwrite")
                .format("delta")
                .option("overwriteSchema", True)
                .saveAsTable(table_name)
            )

            row_count = spark.table(table_name).count()
            created_tables.append((table_name, row_count))
            print(f"Created table: {table_name} ({row_count:,} rows)")

        print("\nLakehouse table materialization complete")
        print(f"  Created tables: {len(created_tables)}")
        print(f"  Skipped files:  {len(skipped_files)}")

        if created_tables:
            print("\nTable preview:")
            for name, count in created_tables:
                print(f"  - {name}: {count:,} rows")

        if skipped_files:
            print("\nSkipped CSV files:")
            for name in skipped_files:
                print(f"  - {name}")

    except Exception as e:
        print(f"Lakehouse table materialization failed: {e}")
        import traceback

        traceback.print_exc()

In [ ]:
print("\n" + "=" * 70)
print("STEP 1C: VALIDATE SEMANTIC MODEL TABLE BINDINGS")
print("=" * 70)

try:
    semantic_model_path = Path(LAKEHOUSE_SEMANTIC_DIR) / "semantic_model.json"

    if not semantic_model_path.exists():
        raise FileNotFoundError(
            f"Semantic model file not found: {semantic_model_path}. "
            "Run Step 1 with INCLUDE_SEMANTIC_REPORT = True first."
        )

    semantic_model = json.loads(semantic_model_path.read_text(encoding="utf-8"))
    semantic_tables = semantic_model.get("tables", [])

    if not semantic_tables:
        raise ValueError("semantic_model.json contains no table definitions")

    print(f"Semantic model: {semantic_model.get('modelName', 'UnnamedModel')}")
    print(f"Table bindings to validate: {len(semantic_tables)}")

    existing_bindings = []
    missing_bindings = []

    for table_def in semantic_tables:
        file_ref = str(table_def.get("file", "")).strip()
        if not file_ref:
            missing_bindings.append(("", "", "Missing file reference in semantic model"))
            continue

        base_name = Path(file_ref).stem.replace("-", "_").replace(" ", "_").lower()
        expected_table_name = f"{LAKEHOUSE_TABLE_PREFIX}{base_name}".lower()

        exists = spark.catalog.tableExists(expected_table_name)
        if exists:
            row_count = spark.table(expected_table_name).count()
            existing_bindings.append((expected_table_name, file_ref, row_count))
            print(f"OK  {expected_table_name} ({row_count:,} rows) <- {file_ref}")
        else:
            missing_bindings.append((expected_table_name, file_ref, "Table not found"))
            print(f"MISSING  {expected_table_name} <- {file_ref}")

    print("\nValidation summary:")
    print(f"  Existing tables: {len(existing_bindings)}")
    print(f"  Missing tables:  {len(missing_bindings)}")

    validation_output = {
        "validatedAtUtc": datetime.now(timezone.utc).isoformat(),
        "semanticModelPath": str(semantic_model_path),
        "tablePrefix": LAKEHOUSE_TABLE_PREFIX,
        "existing": [
            {"tableName": name, "file": file_ref, "rowCount": count}
            for name, file_ref, count in existing_bindings
        ],
        "missing": [
            {"tableName": name, "file": file_ref, "reason": reason}
            for name, file_ref, reason in missing_bindings
        ],
        "isReadyForReporting": len(missing_bindings) == 0,
    }

    semantic_validation_path = Path(LAKEHOUSE_SEMANTIC_DIR) / "semantic_validation.json"
    semantic_validation_path.write_text(json.dumps(validation_output, indent=2), encoding="utf-8")
    print(f"Validation report written: {semantic_validation_path}")

    if missing_bindings:
        print("\nReporting readiness: BLOCKED")
        print("Create the missing tables first (run Step 1b), then re-run this validation cell.")
    else:
        print("\nReporting readiness: READY")

except Exception as e:
    print(f"Semantic model validation failed: {e}")
    import traceback

    traceback.print_exc()

## Step 2: Import to Target Purview

First run a dry-run import to validate, then optionally apply changes.

In [ ]:
print("\n" + "="*70)
print("STEP 2: IMPORT TO TARGET PURVIEW")
print("="*70)

try:
    # Load source manifest
    manifest_data = read_json(SOURCE_MANIFEST_PATH)
    from purview_migration.models import MigrationManifest
    manifest = MigrationManifest.from_dict(manifest_data)

    # Dry-run import
    if RUN_IMPORT_DRY_RUN:
        print(f"\nRunning DRY-RUN import to: {TARGET_PURVIEW_ACCOUNT}")
        result_dryrun = import_manifest(TARGET_PURVIEW_ACCOUNT, manifest, dry_run=True)
        print("Dry-run complete")
        print(f"  Skipped (would process): {result_dryrun.skipped}")
        print(f"  Warnings: {len(result_dryrun.warnings)}")
        if result_dryrun.warnings:
            for w in result_dryrun.warnings[:3]:
                print(f"    - {w}")

    # Apply import if requested
    if RUN_IMPORT_APPLY:
        print(f"\nApplying import to: {TARGET_PURVIEW_ACCOUNT}")
        result_apply = import_manifest(TARGET_PURVIEW_ACCOUNT, manifest, dry_run=False)
        print("Import completed")
        print(f"  Created: {result_apply.created}")
        print(f"  Updated: {result_apply.updated}")
        print(f"  Failed: {result_apply.failed}")

        # Save result status
        write_json(TARGET_STATUS_PATH, {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "target_account": TARGET_PURVIEW_ACCOUNT,
            "result": result_apply.as_dict()
        })
        print(f"Status saved to: {TARGET_STATUS_PATH}")
    else:
        print("\nImport --apply not enabled. Skipping actual import.")
        print("To apply: Set RUN_IMPORT_APPLY = True in configuration cell")

except Exception as e:
    print(f"Import failed: {e}")
    import traceback
    traceback.print_exc()

## Step 3: Generate Relink Plan

Create a name-based mapping plan that links artifacts between source and target.

In [ ]:
print("\n" + "="*70)
print("STEP 3: GENERATE RELINK PLAN")
print("="*70)

try:
    # Load manifest
    manifest_data = read_json(SOURCE_MANIFEST_PATH)
    manifest = MigrationManifest.from_dict(manifest_data)

    # Generate relink plan
    plan = build_relink_plan(manifest)
    print("\nRelink plan generated")

    # Save plan
    write_json(RELINK_PLAN_PATH, plan)
    print(f"Plan saved to: {RELINK_PLAN_PATH}")

    # Display summary
    print("\nRelink plan summary:")
    for artifact_type, count in plan["summary"].items():
        print(f"  {artifact_type:.<30} {count}")

    # Show notes
    if plan.get("notes"):
        print("\nPlan notes:")
        for note in plan["notes"]:
            print(f"  - {note}")

except Exception as e:
    print(f"Relink plan generation failed: {e}")
    import traceback
    traceback.print_exc()

## Step 4: Apply Relink Plan

Validate and optionally apply the relink mappings to create/link artifacts in target.

In [ ]:
print("\n" + "="*70)
print("STEP 4: APPLY RELINK PLAN")
print("="*70)

try:
    # Load relink plan
    plan = read_json(RELINK_PLAN_PATH)

    # Dry-run relink
    if RUN_RELINK_DRY_RUN:
        print(f"\nRunning DRY-RUN relink against: {TARGET_PURVIEW_ACCOUNT}")
        result_dryrun = apply_relink_plan(
            TARGET_PURVIEW_ACCOUNT,
            plan,
            dry_run=True,
            max_entity_validation=MAX_ENTITY_VALIDATION
        )
        print("Dry-run complete")
        print(f"  Linked:     {result_dryrun.linked}")
        print(f"  Unresolved: {result_dryrun.unresolved}")
        print(f"  Skipped:    {result_dryrun.skipped}")
        print(f"  Failed:     {result_dryrun.failed}")
        if result_dryrun.warnings:
            print(f"\nWarnings: {len(result_dryrun.warnings)}")
            for w in result_dryrun.warnings[:3]:
                print(f"    - {w}")

    # Apply relink if requested
    if RUN_RELINK_APPLY:
        print(f"\nApplying relink to: {TARGET_PURVIEW_ACCOUNT}")
        result_apply = apply_relink_plan(
            TARGET_PURVIEW_ACCOUNT,
            plan,
            dry_run=False,
            max_entity_validation=MAX_ENTITY_VALIDATION
        )
        print("Relink completed")
        print(f"  Created:    {result_apply.created}")
        print(f"  Linked:     {result_apply.linked}")
        print(f"  Unresolved: {result_apply.unresolved}")
        print(f"  Failed:     {result_apply.failed}")

        # Save updated plan status
        write_json(RELINK_STATUS_PATH, plan)
        print(f"Updated plan saved to: {RELINK_STATUS_PATH}")
    else:
        print("\nRelink --apply not enabled. Skipping actual relink.")
        print("To apply: Set RUN_RELINK_APPLY = True in configuration cell")

except Exception as e:
    print(f"Relink apply failed: {e}")
    import traceback
    traceback.print_exc()

## Step 5: Generate Status Reports

Export relink results grouped by outcome (JSON/CSV) for easy analysis.

In [ ]:
print("\n" + "="*70)
print("STEP 5: GENERATE STATUS REPORTS")
print("="*70)

try:
    if not GENERATE_REPORTS:
        print("\nReport generation disabled")
        print("To enable: Set GENERATE_REPORTS = True in configuration cell")
    else:
        # Load latest plan (with or without status updates)
        plan_path = RELINK_STATUS_PATH if Path(RELINK_STATUS_PATH).exists() else RELINK_PLAN_PATH
        plan = read_json(plan_path)

        # Generate JSON report
        if REPORT_FORMAT in ["json", "both"]:
            print("\nGenerating JSON report...")
            export_report(plan, REPORT_JSON_PATH, format_type="json")
            print(f"JSON report: {REPORT_JSON_PATH}")

        # Generate CSV reports
        if REPORT_FORMAT in ["csv", "both"]:
            print("\nGenerating CSV reports...")
            export_report(plan, REPORT_CSV_PREFIX, format_type="csv")

            # List CSV files created
            csv_dir = Path(REPORT_CSV_PREFIX).parent
            csv_files = list(csv_dir.glob("relink-*.csv"))
            print(f"Created {len(csv_files)} CSV files:")
            for csv_file in sorted(csv_files):
                size = csv_file.stat().st_size
                print(f"  - {csv_file.name} ({size:,} bytes)")

        print("\nReports generated and saved to Files area")
        print(f"Location: {REPORTS_DIR}")

        # Display report summary
        if REPORT_FORMAT in ["json", "both"] and Path(REPORT_JSON_PATH).exists():
            with open(REPORT_JSON_PATH) as f:
                report = json.load(f)
            print("\nReport summary:")
            for status, count in report.get("summary", {}).get("byStatus", {}).items():
                print(f"  {status:.<20} {count}")

except Exception as e:
    print(f"Report generation failed: {e}")
    import traceback
    traceback.print_exc()

## Step 6: Review and Manage Files in Lakehouse

List, examine, and manage all migration artifacts stored in the Files area.

In [ ]:
print("\n" + "=" * 70)
print("STEP 6: LIST MIGRATION ARTIFACTS IN FILES AREA")
print("=" * 70)


def list_directory_contents(directory, max_depth=3, current_depth=0):
    """Recursively list directory contents with file sizes."""
    if current_depth >= max_depth:
        return

    try:
        dir_path = Path(directory)
        if not dir_path.exists():
            print(f"  (Directory does not exist yet: {directory})")
            return

        items = sorted(dir_path.iterdir())
        for item in items:
            indent = "  " * (current_depth + 1)
            if item.is_file():
                size = item.stat().st_size
                size_str = f"{size:,} B" if size < 1024 else f"{size / 1024:.1f} KB"
                print(f"{indent}- {item.name:.<45} {size_str:>10}")
            elif item.is_dir():
                print(f"{indent}+ {item.name}/")
                list_directory_contents(item, max_depth, current_depth + 1)
    except Exception as e:
        print(f"  Error listing directory: {e}")


# List migration artifacts
for category, dir_path in [
    ("Manifests", MANIFESTS_DIR),
    ("Plans", PLANS_DIR),
    ("Reports", REPORTS_DIR),
    ("Lakehouse Package", LAKEHOUSE_PACKAGE_DIR),
    ("Lakehouse JSON", LAKEHOUSE_JSON_DIR),
    ("Lakehouse Tables", LAKEHOUSE_TABLES_DIR),
    ("Lakehouse Semantic", LAKEHOUSE_SEMANTIC_DIR),
    ("Logs", LOGS_DIR),
]:
    print(f"\n{category}:")
    list_directory_contents(dir_path, max_depth=2)

print("\n" + "=" * 70)
print("FILE STATISTICS")
print("=" * 70)

try:
    artifact_dirs = {
        "Manifests": MANIFESTS_DIR,
        "Plans": PLANS_DIR,
        "Reports": REPORTS_DIR,
        "Lakehouse Package": LAKEHOUSE_PACKAGE_DIR,
    }

    total_files = 0
    total_size = 0

    for category, dir_path in artifact_dirs.items():
        dir_obj = Path(dir_path)
        if dir_obj.exists():
            files = list(dir_obj.rglob("*"))
            file_count = len([f for f in files if f.is_file()])
            total_size_cat = sum(f.stat().st_size for f in files if f.is_file())

            total_files += file_count
            total_size += total_size_cat

            size_str = (
                f"{total_size_cat / (1024 * 1024):.2f} MB"
                if total_size_cat > 1024 * 1024
                else f"{total_size_cat / 1024:.1f} KB"
            )
            print(f"{category:.<30} {file_count:>3} files, {size_str:>10}")

    total_size_str = (
        f"{total_size / (1024 * 1024):.2f} MB"
        if total_size > 1024 * 1024
        else f"{total_size / 1024:.1f} KB"
    )
    print(f"{'Total':.<30} {total_files:>3} files, {total_size_str:>10}")

except Exception as e:
    print(f"Error getting statistics: {e}")

## Summary and Next Steps

Purview migration workflow now includes Lakehouse analytics packaging and managed table materialization.

### Outputs generated in Files

- Manifests: source export and import status files
- Plans: relink plan and relink execution status
- Reports: relink JSON and CSV outcomes
- Lakehouse package:
  - json folder with artifact JSON files
  - tables folder with CSVs for collections, data sources, scans, entities, and summary
  - semantic folder with semantic_model.json, report_spec.json, and capture summaries

### Managed Lakehouse tables created

When `CREATE_LAKEHOUSE_TABLES_FROM_CSV = True`, Step 1b creates Delta tables with names like:
- `purview_migration_collections`
- `purview_migration_datasources`
- `purview_migration_scans`
- `purview_migration_entities`
- `purview_migration_artifact_summary`

### Recommended follow-up actions

1. Review unresolved relink items in reports.
2. Connect semantic model/report setup to the managed Lakehouse tables.
3. Validate that assets, scans, and collections counts match expected source totals.
4. Trigger representative scans in the target account and confirm lineage and governance mappings.

## Utility: Clean Up Old Migration Files

Optional cell to remove old migration artifacts and start fresh.

In [ ]:
# ============================================================================
# CLEANUP: Uncomment and run to delete old migration files
# ============================================================================
# This is optional and useful if you're running multiple migrations
# and want to clean up previous artifacts.

CLEANUP_ENABLED = False  # Set to True to enable cleanup

if CLEANUP_ENABLED:
    print("Cleaning up old migration files...")
    
    try:
        # Optional: remove entire purview_migration directory and start fresh
        migration_root = Path(LAKEHOUSE_FILES) / "purview_migration"
        if migration_root.exists():
            import shutil
            shutil.rmtree(migration_root)
            print(f"✓ Deleted: {migration_root}")
            
        # Recreate directory structure
        ensure_dirs()
        print("✓ Directory structure recreated")
        
    except Exception as e:
        print(f"✗ Cleanup failed: {e}")
else:
    print("⏭ Cleanup disabled")
    print("  To enable cleanup: Set CLEANUP_ENABLED = True and re-run this cell")